# Prepare dengue + rainfall monthly data

Imports, input/output paths, and the name mapping used to make state and district names consistent across the two files.


In [ ]:
import pandas as pd

DENGUE_FILE = "original/Final_data.csv"
RAINFALL_FILE = "original/daily-rainfall-data-district-level.csv"
OUTPUT_FILE = "data_used/dengue_monthly.csv"

NAME_MAPPING = {
    # District spellings
    "pudukkottai": "pudukottai",
    # The rainfall file combines these two former states
    "dadra and nagar haveli": "the dadra and nagar haveli and daman and diu",
    "daman and diu": "the dadra and nagar haveli and daman and diu",
}

print("Dengue file:  ", DENGUE_FILE)
print("Rainfall file:", RAINFALL_FILE)
print("Output file:  ", OUTPUT_FILE)


## 1. Name-cleaning helper

Trim whitespace, lowercase, replace `&` with `and`, collapse extra spaces, then apply the mapping.


In [ ]:
def clean_name(value):
    if pd.isna(value):
        return value
    name = str(value).strip().lower().replace("&", "and")
    name = " ".join(name.split())
    return NAME_MAPPING.get(name, name)

for example in ["Pudukkottai", "Dadra and Nagar Haveli", "Jammu & Kashmir"]:
    print(example, "->", clean_name(example))


## 2. Load dengue data

Read the raw dengue file, drop the unnamed index column, and keep only rows where `Disease` is `Dengue`.


In [ ]:
dengue = pd.read_csv(DENGUE_FILE)
print("Rows loaded:", len(dengue))

dengue = dengue.drop(columns=[c for c in dengue.columns if c.startswith("Unnamed")])
dengue = dengue[dengue["Disease"] == "Dengue"].copy()

print("Rows with Disease == 'Dengue':", len(dengue))
dengue.head()


## 3. Clean dengue rows

Convert `Cases` to numeric, drop invalid values such as `3/1`, clean state and district names, and remove exact duplicate rows. Row counts are printed after every step.


In [ ]:
print("Starting rows:", len(dengue))

dengue["Cases"] = pd.to_numeric(dengue["Cases"], errors="coerce")
before = len(dengue)
dengue = dengue.dropna(subset=["Cases"])
print("After dropping invalid Cases:", len(dengue), f"(dropped {before - len(dengue)})")

dengue["state"] = dengue["state_ut"].map(clean_name)
dengue["district"] = dengue["district"].map(clean_name)
print("After cleaning state/district names:", len(dengue))

before = len(dengue)
dengue = dengue.drop_duplicates()
print("After dropping exact duplicates:", len(dengue), f"(dropped {before - len(dengue)})")

dengue[["state", "district", "Cases", "day", "mon", "year"]].head()


## 4. Fix repeated outbreak reports

Rows with the same state, district, day, month, and year are repeated reports of one outbreak. Keep only the highest-`Cases` row in each group.


In [ ]:
before = len(dengue)
dengue = (
    dengue
    .sort_values("Cases", ascending=False)
    .drop_duplicates(subset=["state", "district", "day", "mon", "year"], keep="first")
    .copy()
)

print("Before outbreak fix:", before)
print("After keeping highest Cases per outbreak:", len(dengue))

dengue[["state", "district", "day", "mon", "year", "Cases"]].head()


## 5. Build monthly dengue cases

Create a real date from `day`, `mon`, and `year`, then sum `Cases` by state and month. Missing state-month combinations are left missing, not filled with zero.


In [ ]:
dengue["date"] = pd.to_datetime(
    dengue[["year", "mon", "day"]].rename(columns={"mon": "month"}),
    errors="coerce",
)
print("Rows with invalid date:", dengue["date"].isna().sum())
dengue = dengue.dropna(subset=["date"])

dengue["month"] = dengue["date"].dt.strftime("%Y-%m")

monthly_cases = (
    dengue
    .groupby(["state", "month"], as_index=False)["Cases"]
    .sum()
    .rename(columns={"Cases": "cases"})
    .sort_values(["state", "month"])
)

print("Monthly state rows:", len(monthly_cases))
monthly_cases.head()


## 6. Load rainfall data

Read only `date`, `state_name`, `actual`, and `normal`, clean the state names, and drop rows where either `actual` or `normal` is missing.


In [ ]:
rain = pd.read_csv(
    RAINFALL_FILE,
    usecols=["date", "state_name", "actual", "normal"],
)
print("Rows loaded:", len(rain))

rain["state"] = rain["state_name"].map(clean_name)
print("After cleaning state names:", len(rain))

before = len(rain)
rain["actual"] = pd.to_numeric(rain["actual"], errors="coerce")
rain["normal"] = pd.to_numeric(rain["normal"], errors="coerce")
rain = rain.dropna(subset=["actual", "normal"])
print("After dropping missing actual/normal:", len(rain), f"(dropped {before - len(rain)})")

rain.head()


## 7. Build monthly rainfall

Create `month` (`YYYY-MM`) and sum `actual` and `normal` for each state-month. Departure is missing when `normal_sum` is zero.


In [ ]:
rain["month"] = pd.to_datetime(rain["date"], errors="coerce").dt.strftime("%Y-%m")
print("Rows with invalid date:", rain["month"].isna().sum())
rain = rain.dropna(subset=["month"])

rain_monthly = (
    rain
    .groupby(["state", "month"], as_index=False)
    .agg(actual_sum=("actual", "sum"), normal_sum=("normal", "sum"))
)

rain_monthly["departure"] = (
    (rain_monthly["actual_sum"] - rain_monthly["normal_sum"])
    / rain_monthly["normal_sum"].where(rain_monthly["normal_sum"] != 0)
)

print("Monthly state rows:", len(rain_monthly))
rain_monthly.head()


## 8. Attach previous-month rainfall

For each state and month `M`, look up rainfall from month `M - 1` and inner-join it onto the dengue monthly table.


In [ ]:
rain_monthly["prev_month"] = (
    pd.to_datetime(rain_monthly["month"] + "-01") - pd.DateOffset(months=1)
).dt.strftime("%Y-%m")

rain_prev = (
    rain_monthly[["state", "prev_month", "actual_sum", "normal_sum", "departure"]]
    .rename(columns={
        "prev_month": "month",
        "actual_sum": "rain_prev_actual",
        "normal_sum": "rain_prev_normal",
        "departure": "rain_prev_departure",
    })
)

merged = monthly_cases.merge(rain_prev, on=["state", "month"], how="inner")

print("Dengue monthly rows before join:", len(monthly_cases))
print("Rows after inner join:", len(merged))
merged.head()


## 9. Check coverage and name matches

Show how many matched rows each state has and list state names that occur in only one of the two files.


In [ ]:
print("Matched rows per state:")
print(merged.groupby("state").size().sort_values(ascending=False).to_string())

dengue_states = set(monthly_cases["state"])
rain_states = set(rain_monthly["state"])

print("\nOnly in dengue file:", sorted(dengue_states - rain_states))
print("Only in rainfall file:", sorted(rain_states - dengue_states))


## 10. Save the monthly table

Write the final selected columns to `processed/dengue_monthly.csv`.


In [ ]:
output = merged[
    ["state", "month", "cases", "rain_prev_actual", "rain_prev_normal", "rain_prev_departure"]
].sort_values(["state", "month"])

output.to_csv(OUTPUT_FILE, index=False)

print("Saved rows:", len(output))
print("Saved columns:", list(output.columns))
print("Saved to:", OUTPUT_FILE)
